In [ ]:
import os
import warnings
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from utils.dataset import CreateDataset
import imageio.v2 as imageio
from tqdm import tqdm

from datetime import datetime

warnings.filterwarnings("ignore")


image_set = "/home/zhangjian/Repositories/Datasets/I2I/IXI/"
contrast1 = "t1"
contrast2 = "t2"

dataset = CreateDataset(
    phase="train", input_path=image_set, contrast1=contrast1, contrast2=contrast2
)
data_loader = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=4)

val_dataset = CreateDataset(
    phase="val", input_path=image_set, contrast1=contrast1, contrast2=contrast2
)
val_data = DataLoader(val_dataset, batch_size=1, shuffle=False)

In [ ]:
from model import SFModel

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

lr = 1e-4
w_decay = 0.01
step = 10
gamma = 0.1

model = SFModel()
optm = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=w_decay)
scheduler = torch.optim.lr_scheduler.StepLR(optm, step_size=step, gamma=gamma)
# 载入模型
# model.load_state_dict(
#     torch.load(
#         "/home/zhangjian/Repositories/DSLFuse2/log/IXI/t1_t2/01-09-15-24/modelsave/modelsynt.pth"
#     )
# )
model = model.to(device)

In [ ]:
# import csv

# from utils.matrics import compute_similarity_metrics_arr
# from utils.misc import tensors2imgarr
# from prettytable import PrettyTable

# from utils.loss import Fusionloss, GradLoss, DcmpLoss, SSIMLoss
# savedir = os.path.join(
#     "/home/zhangjian/Repositories/DSLFuse2/log/IXI/",
#     contrast1 + "_" + contrast2,
#     time_train,
# )

# imsavedir = os.path.join(savedir, "imsave")
# if not os.path.exists(imsavedir):
#     os.makedirs(imsavedir)
# modelsave = os.path.join(savedir, "modelsave")
# if not os.path.exists(modelsave):
#     os.makedirs(modelsave)
# resdir = os.path.join(savedir, "result")
# if not os.path.exists(resdir):
#     os.makedirs(resdir)

# criteria_mse = nn.MSELoss().cuda()
# criteria_l1 = nn.L1Loss().cuda()
# criteria_ssim = SSIMLoss().cuda()
# criteria_grad = GradLoss().cuda()
# criteria_fuse = Fusionloss().cuda()
# criteria_dcmp = DcmpLoss().cuda()


# num_epochs = 60

# ssim_coeff = 0.1
# dcmp_coeff = 0.1

# for epoch in range(num_epochs):
#     model.train()

#     loss_epoch = 0

#     for a, b in tqdm(data_loader):
#         optm.zero_grad()
#         ra, rb = a.to(device), b.to(device)

#         comm_feat_a, spec_feat_a = model.comm_enc(ra), model.spec_enc(ra)
#         comm_feat_b, spec_feat_b = model.comm_enc(rb), model.spec_enc(rb)

#         loss_dcmp = criteria_dcmp(comm_feat_a, comm_feat_b, spec_feat_a, spec_feat_b)

#         fb = model.dec(torch.cat((comm_feat_a, spec_feat_a), dim=1))
#         fa = model.dec(torch.cat((comm_feat_b, spec_feat_b), dim=1))

#         loss_a = criteria_l1(ra, fa) + ssim_coeff * criteria_ssim(ra, fa)
#         loss_b = criteria_l1(rb, fb) + ssim_coeff * criteria_ssim(rb, fb)
#         loss = loss_a + loss_b + dcmp_coeff * loss_dcmp

#         ab_all = np.hstack(tensors2imgarr((ra, fa, rb, fb)))
#         imageio.imwrite("train_show" + ".png", ab_all)

#         loss_epoch += loss.item()

#         loss.backward()
#         optm.step()

#     print("loss =", loss_epoch / len(data_loader))

#     with torch.no_grad():
#         model.eval()

#         metric_a = []
#         metric_b = []

#         for i, ab in enumerate(tqdm(val_data)):
#             a, b = ab
#             ra, rb = a.to(device), b.to(device)

#             comm_feat_a, spec_feat_a = model.comm_enc(ra), model.spec_enc(ra)
#             comm_feat_b, spec_feat_b = model.comm_enc(rb), model.spec_enc(rb)

#             fb = model.dec(torch.cat((comm_feat_a, spec_feat_a), dim=1))
#             fa = model.dec(torch.cat((comm_feat_b, spec_feat_b), dim=1))

#             ra, fa, rb, fb = tensors2imgarr((ra, fa, rb, fb))
#             imageio.imwrite(
#                 os.path.join(savedir, "imsave/ab_" + str(i) + ".png"),
#                 np.hstack((ra, fa, rb, fb)),
#             )
#             metric_a.append(compute_similarity_metrics_arr(ra, fa))
#             metric_b.append(compute_similarity_metrics_arr(rb, fb))

#         metric_a = np.array(metric_a)
#         metric_a_mean = np.mean(metric_a, axis=0)
#         metric_a_std = np.std(metric_a, axis=0)

#         metric_b = np.array(metric_b)
#         metric_b_mean = np.mean(metric_b, axis=0)
#         metric_b_std = np.std(metric_b, axis=0)

#         # 打印 metric
#         table = PrettyTable(["psnr", "psnr_nbg", "ssim", "mse", "mae", "nmi", "vif"])
#         table.add_row(["a", "", "", "", "", "", ""])
#         table.add_row(metric_a_mean)
#         table.add_row(metric_a_std)
#         table.add_row(["b", "", "", "", "", "", ""])
#         table.add_row(metric_b_mean)
#         table.add_row(metric_b_std)
#         print(table)

#         # 保存 metric
#         with open(os.path.join(resdir, "a_epoch_" + str(epoch + 1) + ".csv"), "a") as f:
#             writer = csv.writer(f)
#             writer.writerows(metric_a)
#             writer.writerow(metric_a_mean)
#             writer.writerow(metric_a_std)

#         with open(os.path.join(resdir, "b_epoch_" + str(epoch + 1) + ".csv"), "a") as f:
#             writer = csv.writer(f)
#             writer.writerows(metric_b)
#             writer.writerow(metric_b_mean)
#             writer.writerow(metric_b_std)

#         print("\n====================", epoch + 1, "====================\n")
#     if scheduler is not None:
#         scheduler.step()

# torch.save(
#     model.state_dict(),
#     os.path.join(savedir, "modelsave/modelsynt.pth"),
# )

In [ ]:
import csv

from utils.matrics import Evaluator
from utils.misc import tensors2imgarr
from prettytable import PrettyTable
from os.path import join

from utils.loss import Fusionloss, GradLoss, DcmpLoss, SSIMLoss


time_train = datetime.now().strftime("%m-%d-%H-%M")

savedir = os.path.join(
    "/home/zhangjian/Repositories/DSLFuse2/log/IXI/",
    "fuse_" + contrast1 + "_" + contrast2,
    time_train,
)

imsavedir = os.path.join(savedir, "imsave")
if not os.path.exists(imsavedir):
    os.makedirs(imsavedir)
modelsave = os.path.join(savedir, "modelsave")
if not os.path.exists(modelsave):
    os.makedirs(modelsave)
resdir = os.path.join(savedir, "result")
if not os.path.exists(resdir):
    os.makedirs(resdir)

criteria_mse = nn.MSELoss().cuda()
criteria_l1 = nn.L1Loss().cuda()
criteria_ssim = SSIMLoss().cuda()
criteria_grad = GradLoss().cuda()
criteria_fuse = Fusionloss().cuda()
criteria_dcmp = DcmpLoss().cuda()


num_epochs = 60

ssim_coeff = 0.1
dcmp_coeff = 0.1
fuse_coeff = 0.1

for epoch in range(num_epochs):
    model.train()

    loss_epoch = 0

    for a, b in tqdm(data_loader):
        optm.zero_grad()
        ra, rb = a.to(device), b.to(device)

        comm_feat_a, spec_feat_a = model.comm_enc(ra), model.spec_enc(ra)
        comm_feat_b, spec_feat_b = model.comm_enc(rb), model.spec_enc(rb)

        loss_dcmp = criteria_dcmp(comm_feat_a, comm_feat_b, spec_feat_a, spec_feat_b)

        comm_feat_fuse = model.comm_ffm(torch.cat((comm_feat_a, comm_feat_b), dim=1))
        spec_feat_fuse = model.spec_ffm(torch.cat((spec_feat_a, spec_feat_b), dim=1))

        fuse_ab_all = model.dec(torch.cat((comm_feat_fuse, spec_feat_fuse), dim=1))

        loss_fuse, _, _ = criteria_fuse(ra, rb, fuse_ab_all)

        loss = loss_fuse + dcmp_coeff * loss_dcmp

        ab_all = np.hstack(tensors2imgarr((ra, rb, fuse_ab_all)))
        imageio.imwrite("train_show" + ".png", ab_all)

        loss_epoch += loss.item()

        loss.backward()
        optm.step()

    print("loss =", loss_epoch / len(data_loader))

    with torch.no_grad():
        model.eval()

        for i, ab in enumerate(tqdm(val_data)):
            a, b = ab
            ra, rb = a.to(device), b.to(device)

            comm_feat_a, spec_feat_a = model.comm_enc(ra), model.spec_enc(ra)
            comm_feat_b, spec_feat_b = model.comm_enc(rb), model.spec_enc(rb)

            comm_feat_fuse = model.comm_ffm(
                torch.cat((comm_feat_a, comm_feat_b), dim=1)
            )
            spec_feat_fuse = model.spec_ffm(
                torch.cat((spec_feat_a, spec_feat_b), dim=1)
            )

            fuse_ab_all = model.dec(torch.cat((comm_feat_fuse, spec_feat_fuse), dim=1))

            ra, rb, fuse_ab_all = tensors2imgarr((ra, rb, fuse_ab_all))
            img_prefix = "imsave/img_" + str(i)
            imageio.imwrite(join(savedir, img_prefix + "_a.png"), ra)
            imageio.imwrite(join(savedir, img_prefix + "_b.png"), rb)
            imageio.imwrite(join(savedir, img_prefix + "_f.png"), fuse_ab_all)

    metric_fuse = []
    for i in range(len(val_dataset)):
        img_prefix = "imsave/img_" + str(i)
        ra = Evaluator.image_read_cv2(join(savedir, img_prefix + "_a.png"), "GRAY")
        rb = Evaluator.image_read_cv2(join(savedir, img_prefix + "_b.png"), "GRAY")
        f_ab = Evaluator.image_read_cv2(join(savedir, img_prefix + "_f.png"), "GRAY")
        metric_fuse.append(
            np.array(
                [
                    Evaluator.EN(f_ab),
                    Evaluator.SD(f_ab),
                    Evaluator.SF(f_ab),
                    Evaluator.MI(f_ab, ra, rb),
                    Evaluator.SCD(f_ab, ra, rb),
                    Evaluator.VIFF(f_ab, ra, rb),
                    Evaluator.Qabf(f_ab, ra, rb),
                    Evaluator.SSIM(f_ab, ra, rb),
                ]
            )
        )

    metric_fuse = np.array(metric_fuse)
    metric_fuse_mean = np.mean(metric_fuse, axis=0)
    metric_fuse_std = np.std(metric_fuse, axis=0)

    # 打印 metric
    table = PrettyTable(["EN", "SD", "SF", "MI", "SCD", "VIFF", "Qabf", "SSIM"])
    table.add_row(metric_fuse_mean)
    table.add_row(metric_fuse_std)

    print(table)

    # 保存 metric
    with open(os.path.join(resdir, "fuse_epoch_" + str(epoch + 1) + ".csv"), "a") as f:
        writer = csv.writer(f)
        writer.writerows(metric_fuse)
        writer.writerow(metric_fuse_mean)
        writer.writerow(metric_fuse_std)

    print("\n====================", epoch + 1, "====================\n")
    if scheduler is not None:
        scheduler.step()

torch.save(
    model.state_dict(),
    os.path.join(savedir, "modelsave/modelfuse.pth"),
)